# <b>Linking APC with matched_stops</b>

Since the Dilax data and the APC data use different vehicle ID systems, 
we cannot link them by vehicle ID. Instead, we link them based on time 
and GPS coordinates: for each stop event in matched_stops, we find the 
APC row that occurred at the same place around the same time.

## Setup: Imports And Loading Data

In [1]:
import pandas as pd
import numpy as np

apc = pd.read_csv('../data/raw/apc.csv', sep = ';', encoding = 'latin-1')
matched_stops = pd.read_csv('../data/raw/matched_stops.csv', sep = ';', decimal=',', encoding = 'latin-1')

# Parse times
apc['arrival_dt']           = pd.to_datetime(apc['arrival'], format='%d/%m/%Y %H:%M:%S')
matched_stops['arrival_dt'] = pd.to_datetime(matched_stops['arrivalTime'], format='%m-%d-%Y %I:%M:%S %p')

print(f"apc rows:           {len(apc):,}")
print(f"matched_stops rows: {len(matched_stops):,}")

apc rows:           1,598,279
matched_stops rows: 56,165


## Step 1 - Test linking on a single row

We write a simple function that takes one matched_stops row and searches APC for the same event, based on time (±tolerance) and GPS coordinates (±tolerance).

This is a test on a single row, to see how the linking works in practice.

In [8]:
def find_apc_match(ms_row, time_tol_sec=60, gps_tol=0.001):
    return apc[
        (apc['arrival_dt'].between(
            ms_row['arrival_dt'] - pd.Timedelta(seconds=time_tol_sec),
            ms_row['arrival_dt'] + pd.Timedelta(seconds=time_tol_sec))) &
        (apc['latitude'].between(ms_row['latitude']  - gps_tol, ms_row['latitude']  + gps_tol)) &
        (apc['longitude'].between(ms_row['longitude'] - gps_tol, ms_row['longitude'] + gps_tol))
    ]

# Test on the first matched_stops row
target = matched_stops.iloc[50]
match = find_apc_match(target)
# print(matched_stops.iloc[50])

print(f"Target: {target['arrival_dt']} @ {target['stopLabel']}, Dilax vehicle {target['vehicleNumber']}")
print(f"Matches found: {len(match)}")
print()
print(match[['arrival', 'vehicleCode', 'latitude', 'longitude']].to_string())

Target: 2025-01-01 06:23:31 @ Laholm Sologatan, Dilax vehicle 262
Matches found: 1

                  arrival  vehicleCode   latitude  longitude
3981  01/01/2025 06:23:31         4241  56.505104  13.031932


## Step 2 - Test multiple tolerance combinations on the full dataset

To choose appropriate tolerances, we run the matching across all matched_stops rows for several combinations of time and GPS tolerance, and count how many rows get 0, 1, or 2+ matches.

In [10]:
# Sort APC by time once - allows fast time-window lookup with searchsorted
apc_sorted = apc.sort_values('arrival_dt').reset_index(drop=True)
apc_times  = apc_sorted['arrival_dt'].values
apc_lats   = apc_sorted['latitude'].values
apc_lons   = apc_sorted['longitude'].values

# Test multiple tolerance combinations
combos = [
    (30,  0.001),   # tight
    (60,  0.001),   # current
    (60,  0.0015),  # slightly wider GPS
    (60,  0.002),   # wider GPS
    (90,  0.0015),  # wider time + GPS
    (120, 0.002),   # most generous
]

print(f"{'time_tol':>10} {'gps_tol':>10} {'0 matches':>12} {'1 match':>12} {'2+ matches':>12}")
print("-" * 60)

for time_tol_sec, gps_tol in combos:
    n_matches = []
    for i in range(len(matched_stops)):
        ms_row = matched_stops.iloc[i]
        t = ms_row['arrival_dt'].to_datetime64()
        left  = np.searchsorted(apc_times, t - np.timedelta64(time_tol_sec, 's'))
        right = np.searchsorted(apc_times, t + np.timedelta64(time_tol_sec, 's'))
        if left == right:
            n_matches.append(0)
            continue
        lat_ok = np.abs(apc_lats[left:right] - ms_row['latitude'])  <= gps_tol
        lon_ok = np.abs(apc_lons[left:right] - ms_row['longitude']) <= gps_tol
        n_matches.append((lat_ok & lon_ok).sum())
    
    n = np.array(n_matches)
    print(f"{time_tol_sec:>9}s  {gps_tol:>10.4f}  {(n==0).sum():>12,}  {(n==1).sum():>12,}  {(n>=2).sum():>12,}")

  time_tol    gps_tol    0 matches      1 match   2+ matches
------------------------------------------------------------
       30s      0.0010        40,627        14,885           653
       60s      0.0010        39,801        15,095         1,269
       60s      0.0015        13,719        38,916         3,530
       60s      0.0020         4,028        46,985         5,152
       90s      0.0015        13,169        37,846         5,150
      120s      0.0020         3,347        43,661         9,157
